# TinyFace alignment benchmark — Modal controller
Notebook này chạy trong Jupyter/local notebook và điều khiển Modal GPU; không dùng `google.colab`.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path.cwd()
if not (ROOT/'CVLface-main'/'benchmarks'/'tinyface_alignment'/'modal_app.py').exists():
    ROOT = Path('/content/Landmark-aligmenttest')
REPO = ROOT/'CVLface-main'
DATA = REPO/'tinyface'
print('ROOT=', ROOT)


In [ ]:
# Cài công cụ tải dữ liệu
subprocess.check_call([sys.executable,'-m','pip','install','-q','gdown'])


In [ ]:
# Tải và giải nén TinyFace vào đúng REPO/tinyface
FILE_ID='1xTZc7lNmWN33ECO2AKH6FycGdiqIK7W0'
ROOT.mkdir(parents=True,exist_ok=True)
ZIP=ROOT/'tinyface.zip'
if not (DATA/'Testing_Set'/'Probe').is_dir():
    import gdown
    downloaded=gdown.download(id=FILE_ID,output=str(ZIP),quiet=False)
    if not downloaded or not ZIP.is_file(): raise RuntimeError('Không tải được ZIP. Hãy bật Anyone with the link hoặc đặt ZIP thủ công tại '+str(ZIP))
    import zipfile, tempfile, shutil
    with tempfile.TemporaryDirectory() as td:
        with zipfile.ZipFile(ZIP) as z: z.extractall(td)
        candidates=[Path(td)]+list(Path(td).rglob('*'))
        src=next((p for p in candidates if p.is_dir() and (p/'Testing_Set'/'Probe').is_dir()),None)
        if src is None: raise RuntimeError('ZIP không có Testing_Set/Probe')
        if DATA.exists(): shutil.rmtree(DATA)
        DATA.parent.mkdir(parents=True,exist_ok=True)
        shutil.copytree(src,DATA)
for split in ('Probe','Gallery_Match','Gallery_Distractor'):
    p=DATA/'Testing_Set'/split
    if not p.is_dir(): raise FileNotFoundError(p)
    print(split,len(list(p.glob('*.jpg'))))


In [ ]:
# Upload dataset lên Modal Volume
subprocess.run(['modal','volume','create','cvlface-tinyface-data'],check=False)
subprocess.check_call(['modal','volume','put','cvlface-tinyface-data',str(DATA),'/tinyface'])
print('Uploaded to /data/tinyface on Modal')


In [ ]:
# Chạy bốn pipeline trên GPU Modal
APP=REPO/'benchmarks'/'tinyface_alignment'/'modal_app.py'
cmd=['modal','run',str(APP),'--probes-per-bin','5','--distractors','100','--pipelines','dfa_mobilenet,dfa_resnet50,scrfd10g,mediapipe']
subprocess.check_call(cmd,cwd=str(ROOT))
print('Kết quả: benchmarks/tinyface_alignment/last_modal_result.json')


In [ ]:
# Đọc kết quả và in bảng tóm tắt
import json
result_path=REPO/'benchmarks'/'tinyface_alignment'/'last_modal_result.json'
result=json.loads(result_path.read_text())
for name, value in result.get('pipelines',{}).items():
    print(name, value.get('retrieval'), value.get('summary'))
